In [0]:
%pip install pendulum

In [0]:
import json
from pathlib import Path

import pendulum
import requests

In [0]:
# Detecta el catálogo actual
current_catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]

catalog = current_catalog

schema = "raw"
volume = "usgs_earthquakes"

# Crear schema
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}"
)

# Crear volumen
spark.sql(
    f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}"
)

In [0]:
raw_path = f"/Volumes/{catalog}/{schema}/{volume}"

dbutils.fs.rm(
    raw_path,
    recurse=True
)

print(f"RAW limpiado: {raw_path}")

In [0]:
dbutils.widgets.text(
    "start_date",
    "2023-01-01",
    "Start Date (YYYY-MM-DD)"
)

dbutils.widgets.text(
    "end_date",
    "2023-12-31",
    "End Date (YYYY-MM-DD)"
)

dbutils.widgets.text(
    "output_dir",
    raw_path,
    "Output Directory"
)

dbutils.widgets.text(
    "api_endpoint",
    "https://earthquake.usgs.gov/fdsnws/event/1/query",
    "API Endpoint"
)

dbutils.widgets.text(
    "timeout",
    "60",
    "Time Out"
)

In [0]:
start_date = pendulum.parse(
    dbutils.widgets.get("start_date")
).date()

end_date = pendulum.parse(
    dbutils.widgets.get("end_date")
).date()

output_dir = Path(
    dbutils.widgets.get("output_dir")
)

API_URL = dbutils.widgets.get(
    "api_endpoint"
)

timeout = int(
    dbutils.widgets.get("timeout")
)

print("Start:", start_date)
print("End:", end_date)
print("Output:", output_dir)
print("API:", API_URL)

In [0]:
saved_files = []
failed_dates = []

current_date = start_date

session = requests.Session()

while current_date <= end_date:

    date_str = current_date.to_date_string()

    next_date = current_date.add(days=1)

    next_date_str = next_date.to_date_string()

    # Estructura YYYY/MM/DD
    daily_output_dir = (
        output_dir
        / f"{current_date.year:04d}"
        / f"{current_date.month:02d}"
        / f"{current_date.day:02d}"
    )

    daily_output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    file_path = (
        daily_output_dir
        / "usgs_earthquakes.geojson"
    )

    params = {
        "format": "geojson",
        "starttime": f"{date_str}T00:00:00",
        "endtime": f"{next_date_str}T00:00:00",
        "orderby": "time-asc",
        "limit": 20000
    }

    try:

        response = session.get(
            API_URL,
            params=params,
            timeout=timeout
        )

        response.raise_for_status()

        file_path.write_bytes(
            response.content
        )

        saved_files.append(
            str(file_path)
        )

        print(
            f"{date_str} -> OK"
        )

    except requests.RequestException as e:

        failed_dates.append(
            {
                "date": date_str,
                "error": str(e)
            }
        )

        print(
            f"{date_str} -> ERROR: {e}"
        )

    current_date = next_date

In [0]:
print(
    f"Archivos guardados: {len(saved_files)}"
)

print(
    f"Fechas con error: {len(failed_dates)}"
)

if failed_dates:
    print("Errores:")
    for item in failed_dates:
        print(
            item["date"],
            item["error"]
        )